In [3]:
# GOAL :: Do familail relationship regression (FR-reg)

In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import sys; sys.path.append("/data/jerrylee/pjt/BIGFAM.v.0.1")
from BIGFAM import frreg
import importlib

In [18]:
source = "UKB"

# Step 1. Load data

In [19]:
# parameters
info_fn = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{source}/relative_information/relatives.formatted.info"

In [20]:
# relative information format
df_pair = pd.read_csv(info_fn, sep='\t')
df_pair.head()

,DOR,relationship,volid,relid,volage,relage,volsex,relsex,Erx
0,1,DS,1000094,3653174,65,64,F,F,0.75
1,1,NaN,1000220,1691267,64,64,F,F,NaN
2,1,NaN,1000286,1571411,53,70,F,F,NaN
3,1,NaN,1000295,1045127,60,41,F,F,NaN
4,1,NaN,1000476,3599303,50,51,F,M,NaN


# Step 2. FR-reg

In [21]:
pheno_path = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{source}/phenotype"
frreg_path = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{source}/frreg"

In [22]:
pheno_fns = os.listdir(pheno_path)
len(pheno_fns)

107

## Step 2.1 DOR level

In [23]:
warning_dicts = {}

for fn in tqdm(pheno_fns):
    pheno = fn.split(".")[0]
    pheno_fn = f"{pheno_path}/{fn}"
    df_pheno = pd.read_csv(pheno_fn, sep="\t")
    df_pheno = frreg.remove_outliers(df_pheno, "pheno")
    
    # merge pheno with relatives
    df_mrg = frreg.merge_pheno_info(df_pheno, df_pair)
    
    df_frreg, msgs = frreg.familial_relationship_regression_DOR(df_mrg)
    
    if len(msgs) > 0:
        warning_dicts[pheno] = msgs
        continue
    
    df_frreg.to_csv(
        f"{frreg_path}/DOR/{pheno}.DOR.frreg",
        sep='\t',
        index=False
    )

100%|██████████| 107/107 [02:03<00:00,  1.15s/it]


In [24]:
warning_dicts

{}

In [25]:
df_frreg

,DOR,slope,se,p,n
0,1,0.127574,0.007359,8.855119e-67,18164
1,2,0.038686,0.015377,1.191011e-02,4224
2,3,0.038677,0.006233,5.557872e-10,25700


## Step 2.2 REL level

In [26]:
warning_dicts = {}

for fn in tqdm(pheno_fns):
    pheno = fn.split(".")[0]
    pheno_fn = f"{pheno_path}/{fn}"
    df_pheno = pd.read_csv(pheno_fn, sep="\t")
    df_pheno = frreg.remove_outliers(df_pheno, "pheno")
    
    # merge pheno with relatives
    df_mrg = frreg.merge_pheno_info(df_pheno, df_pair)
    
    df_frreg, msgs = frreg.familial_relationship_regression_REL(df_mrg)
    
    # if len(msgs) > 0:
    #     warning_dicts[pheno] = msgs
    #     continue
    
    # save results
    df_frreg.to_csv(
        f"{frreg_path}/REL/{pheno}.REL.frreg",
        sep='\t',
        index=False
    )

100%|██████████| 107/107 [02:21<00:00,  1.32s/it]
